#               Power Spectral Density (PSD)

Standalone comparison of PSD estimators for a building-vibration acceleration signal.

The notebook uses periodic Hann windows and compares Welch averaging, a periodogram, and a direct one-sided FFT density estimate.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from numpy import loadtxt
from pathlib import Path
import scipy
from scipy import signal
from scipy.signal import welch
plt.rc('font', family='Times New Roman')
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Load vibration signal.
vib_path_candidates = [Path('vib.csv'), Path('PSD') / 'vib.csv']
vib_path = next((path for path in vib_path_candidates if path.exists()), None)
if vib_path is None:
    raise FileNotFoundError('Put vib.csv in the notebook folder or in PSD/.')

vib_sig = np.asarray(loadtxt(vib_path, delimiter=','), dtype=float).squeeze()
if vib_sig.ndim > 1:
    vib_sig = vib_sig[:, 0]

num_samples_vib = len(vib_sig) # number of datapoints in the signal.
fs_vib = 26667 # configured IIS3DWB sampling rate in Hz; change to match the acquisition.
t_vib = np.arange(num_samples_vib) / fs_vib # time vector in seconds.


# Input signal

`vib.csv` should contain one building-sensor acceleration channel in g. Use a synchronized acquisition window when comparing locations.

In [ ]:
# PSD estimation using Welch's method.
fft_length = 4096
seg_length = min(max(1, num_samples_vib // 4), fft_length) # Segment length in samples.
overlap = seg_length // 2 # Overlap between successive segments = half segment length.
win = 'hann' # Periodic Hann window (SciPy FFT-bin convention).
f_psd_w, psd_w = welch(
    vib_sig,
    fs=fs_vib,
    window=win,
    nperseg=seg_length,
    noverlap=overlap,
    nfft=fft_length,
)


In [ ]:
# PSD estimation using periodogram method.
f_psd_p, psd_p = signal.periodogram(vib_sig, fs_vib, window=win, nfft=fft_length)


In [ ]:
# PSD estimation using np.fft.fft on a 100 second window.
fft_window_seconds = 100.0
fft_start_seconds = 0.0
fft_window_samples = int(round(fft_window_seconds * fs_vib))
fft_start_sample = int(round(fft_start_seconds * fs_vib))
fft_stop_sample = fft_start_sample + fft_window_samples

if fft_window_samples <= 0:
    raise ValueError('fft_window_seconds must be positive.')
if fft_start_sample < 0:
    raise ValueError('fft_start_seconds must be non-negative.')
if fft_stop_sample > num_samples_vib:
    available_seconds = num_samples_vib / fs_vib
    raise ValueError(
        f'Need {fft_window_seconds:g} s ({fft_window_samples} samples) for the FFT PSD window, '
        f'but vib.csv only contains {available_seconds:.3f} s ({num_samples_vib} samples).'
    )

vib_fft_window = np.asarray(vib_sig[fft_start_sample:fft_stop_sample], dtype=float)
vib_fft_window = vib_fft_window - np.mean(vib_fft_window)
fft_window = signal.get_window(win, fft_window_samples, fftbins=True)
vib_fft_windowed = vib_fft_window * fft_window

# Keep the complex FFT coefficients, then convert the positive-frequency half to one-sided PSD.
fft_complex = np.fft.fft(vib_fft_windowed)
f_psd_fft = np.fft.rfftfreq(fft_window_samples, d=1.0 / fs_vib)
fft_positive = fft_complex[:f_psd_fft.size]
psd_fft = (np.abs(fft_positive) ** 2) / (fs_vib * np.sum(fft_window ** 2))

if psd_fft.size > 1:
    if fft_window_samples % 2 == 0:
        psd_fft[1:-1] *= 2.0 # Do not double DC or Nyquist.
    else:
        psd_fft[1:] *= 2.0 # Odd-length FFT has no Nyquist bin.

fft_resolution_hz = fs_vib / fft_window_samples
print(
    f'FFT PSD window: {fft_window_seconds:g} s ({fft_window_samples} samples), '
    f'resolution: {fft_resolution_hz:.6g} Hz'
)


In [ ]:
# Plot PSDs.
plt.figure(figsize=(25, 15))
plt.grid(True, which='both')
plt.semilogy(f_psd_w, psd_w, linewidth=2, color='b', label="PSD estimation using Welch's method")
plt.semilogy(f_psd_p, psd_p, linewidth=1, color='c', label='PSD estimation using periodogram method')
plt.semilogy(f_psd_fft, psd_fft, linewidth=1, color='m', label='PSD using np.fft.fft (100 s window)')
plt.xlabel('Frequency (Hz)', fontsize=30)
plt.ylabel('Power spectral density (signal units^2/Hz)', fontsize=30)
plt.tick_params(axis='both', which='major', labelsize=30)
plt.tick_params(axis='both', which='minor', labelsize=30)
plt.legend(fontsize=30)
plt.show()
